---

### eChem course module on:

# [Natural Orbitals](https://kthpanor.github.io/echem/natural-orbitals/)

### [Patrick Norman](https://www.kth.se/profile/panor)

#### [KTH Royal Institute of Technology](https://www.kth.se/), Stockholm, Sweden

---

In [ ]:
import multipsi as mtp
import numpy as np
import veloxchem as vlx

## Restricted calculations

### Formal introduction

The [one-particle density](https://kthpanor.github.io/echem/reduced-density/) and the associated spin-orbital density matrix, $D^\mathrm{SO}$, are related as follows

$$
n(\mathbf{r}) \equiv
\sum_{p,q} \psi^\dagger_p(\mathbf{r}) D_{pq}^\mathrm{SO} \psi_q(\mathbf{r})
$$

A nonzero contribution to the density requires the spin of both orbitals to be identical and the summation can therefore be separated by introducing the $\alpha$- and $\beta$-spin densities and instead run over molecular orbitals

\begin{align*}
n(\mathbf{r}) & =
\sum_{p,q} 
\Big(
\big[
\phi^\alpha_p(\mathbf{r})
\big]^\ast
D_{pq}^\alpha
\phi_q^\alpha(\mathbf{r})
+
\big[
\phi^\beta_p(\mathbf{r})
\big]^\ast
D_{pq}^\beta
\phi_q^\beta(\mathbf{r})
\Big)
\end{align*}

In restricted calculations, we have a common set of MOs for the $\alpha$- and $\beta$-spin orbitals and can write

\begin{align*}
n(\mathbf{r}) & =
\sum_{p,q} 
\phi_p^\ast(\mathbf{r})
D_{pq}
\phi_q(\mathbf{r})
\\
D_{pq} & =
D_{pq}^\alpha + D_{pq}^\beta
\end{align*}

The density matrix is Hermitian and can be diagonalized by a unitary transformation. We get

\begin{align*}
\mathbf{D}^\mathrm{NO} & = 
\mathbf{U}^\dagger
\mathbf{D}
\mathbf{U} \\
\phi_p^\mathrm{NO}(\mathbf{r}) & =
\sum_q
\phi_q(\mathbf{r})
U_{qp}
\end{align*}

In restricted closed (RHF) and open-shell Hartree–Fock (ROHF), the natural orbitals equal the canonical HF orbitals and the occupation numbers equal to 0, 1, and 2 for unoccuppied, singly occupied, and doubly occupied MOs, respectively. In MCSCF, the occupation numbers equal to 0 and 2 for secondary and inactive orbitals, respectively, and anything in between for active orbitals. 

In the NO basis, we have

$$
n(\mathbf{r}) = 
\sum_{p} 
D^\mathrm{NO}_{pp} \,
\big[
\phi_p^\mathrm{NO}(\mathbf{r})
\big]^*
\phi_p^\mathrm{NO}(\mathbf{r})
$$

where the density matrix equals

$$
\mathbf{D}^\mathrm{NO} =
\begin{pmatrix}
2      & \cdots & 0      & 0      & \cdots & 0      & 0      & \cdots & 0 \\
\vdots & \ddots & \vdots & \vdots & \ddots & \vdots & \vdots & \ddots & \vdots \\
0      & \cdots & 2      & 0      & \cdots & 0      & 0      & \cdots & 0 \\
0      & \cdots & 0      & o_1 & \cdots & 0      & 0      & \cdots & 0 \\
\vdots & \ddots & \vdots & \vdots & \ddots & \vdots & \vdots & \ddots & \vdots \\
0      & \cdots & 0      & 0      & \cdots & o_n & 0      & \cdots & 0 \\
0      & \cdots & 0      & 0      & \cdots & 0      & 0      & \cdots & 0 \\
\vdots & \ddots & \vdots & \vdots & \ddots & \vdots & \vdots & \ddots & \vdots \\
0      & \cdots & 0      & 0      & \cdots & 0      & 0      & \cdots & 0 \\
\end{pmatrix}
$$

### Create the system and choose basis set

Let us look at the oxygen molecule with a triplet ground state. 

In [ ]:
mol_str = """
O    0.000   0.000  -0.600
O    0.000   0.000   0.600
"""
molecule = vlx.Molecule.read_molecule_string(mol_str)
basis = vlx.MolecularBasis.read(molecule, "cc-pvdz", ostream=None)

In [ ]:
molecule.show()

### Restricted open-shell HF wave function

In [ ]:
rohf_drv = vlx.ScfRestrictedOpenDriver()

molecule.set_multiplicity(3)
rohf_results = rohf_drv.compute(molecule, basis)

The MOs are occupied by nine α- and seven β-spin electrons.

In [ ]:
print(f"  α-string: {rohf_results["occ_alpha"]}")
print(f"  β-string: {rohf_results["occ_beta"]}")

In [ ]:
rohf_drv.molecular_orbitals.print_orbitals(molecule, basis)

In [ ]:
viewer_scf = vlx.OrbitalViewer()
viewer_scf.plot(molecule, basis, rohf_drv.mol_orbs)

### MCSCF wave function

**Main AO character of MOs**

---
Active orbitals (8 electrons)

- $3\sigma_u = p_z^1 + p_z^2$

- $1\pi_g = p_{x,y}^1 - p_{x,y}^2$

- $1\pi_u = p_{x,y}^1 + p_{x,y}^2$

- $3\sigma_g = p_z^1 - p_z^2$

---
Inactive orbitals (8 electrons)

- $2\sigma_u = 2s^1 - 2s^2$

- $2\sigma_g = 2s^1 + 2s^2$

- $1\sigma_u = 1s^1 - 1s^2$

- $1\sigma_g = 1s^1 + 1s^2$

---

### MCSCF optimization

In [ ]:
mcscf_drv = mtp.McscfDriver()
orbital_space = mtp.OrbSpace(molecule, basis, rohf_drv.mol_orbs)
orbital_space.cas(8, 6)

mcscf_results = mcscf_drv.compute(molecule, basis, orbital_space)

The MCSCF wave function is expressed in terms of the natural orbitals. 

The occupation numbers are (i) equal to 2 for the four inactive orbitals, (ii) close to 2 for $3\sigma_g$ and $1\pi_u$, (iii) close to 1 for $1\pi_g$, and (iv) close to 0 for $3\sigma_u$.

In [ ]:
print(mcscf_results["natural_occupations"][0])

### Correlation energy

In [ ]:
print(f"    ROHF energy: {rohf_results["scf_energy"] : 12.8f}")
print(f"CAS(8,6) energy: {mcscf_results["energies"][0] : 12.8f}")

### Most important configurations

The wave function is predominantly a sum of two determinants with $\alpha$-spin occupation of the $\pi_u$ and $\pi_g$ molecular orbitals, respectively.

In [ ]:
mcscf_results["ci_vectors"].print(0)

### Inspect the orbitals

In [ ]:
viewer_mcscf = vlx.OrbitalViewer()
viewer_mcscf.plot(molecule, basis, orbital_space.molecular_orbitals)

### Density matrix in AO representation

In the atomic orbital (AO) basis, we have

$$
n(\mathbf{r}) = \sum_{\alpha, \beta}
D_{\alpha\beta}^\mathrm{AO} \,
\chi_\alpha^\ast(\mathbf{r})
\chi_\beta(\mathbf{r})
$$

where, in general,

$$
\mathbf{D}^\mathrm{AO} =
\mathbf{C}^* \mathbf{D}^\mathrm{MO} \mathbf{C}^T
$$

and we adopt the NO basis for the MCSCF wave function.

The density matrix in the AO basis is readily available.

In [ ]:
D_ao = mcscf_drv.get_total_density()

By inverting the equation above, we can obtain the density matrix in the MO (or NO) basis. We make use of the identity

$$
\delta_{pq} = \langle \phi_p | \phi_q \rangle = 
\sum_{\alpha, \beta} c^*_{\alpha p} c_{\beta q} S_{\alpha\beta}
$$

or, equivalently,

$$
\mathbf{I} = \mathbf{C}^\dagger \mathbf{S} \mathbf{C}
$$

where $\mathbf{S}$ and $\mathbf{C}$ are the overlap and MO coefficient matrices, respectively.

In [ ]:
# get the NO coefficients and the overlap matrix
C = orbital_space.molecular_orbitals.alpha_to_numpy()
S = rohf_results["S"]

In [ ]:
# perform the inversion
tmp = np.einsum("ai, ab, bc -> ic", C, S, D_ao)
D_mo = np.einsum("fill me in", tmp, S, C)

We expect the density matrix in the NO basis to be diagonal with the occupation numbers on the diagonal.

In [ ]:
np.set_printoptions(precision=4, suppress=True, linewidth=128)
D_mo[:12,:12]

## Unrestricted calculations

### Expressing one set of MOs in the other

In unrestricted calculations, we have separate sets of MOs for the $\alpha$- and $\beta$-spin orbitals. However, one set can be expressed in terms of the other according to

$$
\phi^\beta_s(\mathbf{r}) =
\sum_q \phi^\alpha_q(\mathbf{r}) S_{qs}^{\alpha\beta}
$$

where $S^{\alpha\beta}$ is the overlap between the two sets of orbitals that, in turn, can be expressed in terms of the overlap matrix in the AO basis

$$
S_{pq}^{\alpha\beta} =
\sum_{\delta,\gamma}
\big[c_{\delta p}^\alpha\big]^\ast c^\beta_{\gamma q}
S_{\delta \gamma}
$$

The one-particle density becomes

\begin{align*}
n(\mathbf{r}) & = \sum_{p,q} 
\big[ \phi^\alpha_p(\mathbf{r}) \big]^\ast
D_{pq}
\phi_q^\alpha(\mathbf{r})
\\
D_{pq} & = D_{pq}^\alpha +
\sum_{r,s} 
\big[
S^{\alpha\beta}_{pr}
\big]^\ast
D_{rs}^\beta
S^{\alpha\beta}_{qs}
\end{align*}

and the natural orbitals and occupation numbers are obtained from a diagonalization of this total density matrix.

Let us consider the unrestricted Hartree–Fock state of the oxygen molecule.

In [ ]:
uhf_drv = vlx.ScfUnrestrictedDriver()
uhf_drv.ostream.mute()
uhf_results = uhf_drv.compute(molecule, basis)

In [ ]:
print(f"UDFT energy: {uhf_drv.get_scf_energy() : 14.8f}")

In [ ]:
norb = basis.get_dimensions_of_basis()
n_alpha_electrons = molecule.number_of_alpha_electrons()
n_beta_electrons = molecule.number_of_beta_electrons()

S = uhf_results["S"]
C_alpha = uhf_results["C_alpha"]
C_beta = uhf_results["C_beta"]

D_alpha = np.zeros((norb, norb))
for i in range(n_alpha_electrons):
    D_alpha[i, i] = 1

D_beta = np.zeros((norb, norb))
for i in range(n_beta_electrons):
    D_beta[i, i] = 1

In [ ]:
S_ab = np.einsum("ap, bq, ab -> pq", C_alpha, C_beta, S)
D = D_alpha + np.einsum("pr, rs, qs -> pq", S_ab, D_beta, S_ab)

occ_numbers, U = np.linalg.eigh(D)

In [ ]:
print("Orbital  Occ. number")
for orb, occ in enumerate(list(occ_numbers)[::-1]):
    if occ > 1e-3:
        print(f"{orb + 1 : >2} {occ : 12.4f}")

We can also get the natural orbitals and occupation numbers from the `natural_orbitals` method in VeloxChem.

In [ ]:
natural_orbs = uhf_drv.natural_orbitals(uhf_results)
natural_orbs.occa_to_numpy()

## Summary of natural orbitals

Natural orbitals (NOs) provide a privileged orbital representation by diagonalizing the total one‑particle reduced density matrix of a many‑electron wave function. In this basis, the electronic state is described by orbital occupation numbers that reflect the actual distribution of electrons.

A key strength of natural orbitals is that they offer a compact and physically transparent description of electronic correlation. Orbitals with occupation numbers close to 2 or 0 behave like inactive or virtual orbitals, while those with fractional occupations signal the presence of static (multireference) correlation. This makes natural orbitals a powerful diagnostic tool for identifying near‑degeneracies, bond breaking, and other situations where single‑reference methods fail.

Because they optimally represent the one‑particle density, natural orbitals often lead to faster convergence and reduced active spaces in multiconfigurational methods such as CASSCF, and they provide a natural bridge between correlated wave‑function methods and orbital‑based interpretations. At the same time, it is important to remember that natural orbitals depend on the underlying wave function and therefore inherit its approximations and limitations.

In summary, natural orbitals do not change the physics of a calculation, but they reveal it more clearly—making them an essential conceptual and practical tool in modern electronic‑structure theory.